In [ ]:
from google.colab import files
uploaded = files.upload()


Saving final project cleaning.xlsx to final project cleaning (1).xlsx


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_excel("final project cleaning.xlsx")
df.head()

,EmployeeID,FirstName,LastName,Gender,DateOfBirth,Nationality,MaritalStatus,EducationLevel,Email,DateOfJoining,DateOfLeaving(New),ReasonForLeaving,VoluntaryExit,RecruitmentSource,TimeToHire,CostPerHire
0,110001,Gwyn,Etzel,F,1980-05-17,UK,Single,Associate’s Degree,gwyn.etzel@lumen.com,2024-06-19,NaN,"""null""","""null""",3,45,3000
1,110002,Ressie,Goodwyn,F,1969-04-29,UK,Single,Master’s Degree,ressie.goodwyn@lumen.com,2024-03-29,NaN,"""null""","""null""",2,48,2000
2,110003,Colton,Salzman,M,1997-09-27,USA,Single,Technical Certificate,colton.salzman@lumen.com,2024-03-03,NaN,"""null""","""null""",2,57,4000
3,110004,Marylynn,Ealey,F,1991-08-01,Australia,Single,Technical Certificate,marylynn.ealey@lumen.com,2024-03-11,NaN,"""null""","""null""",4,50,2500
4,110005,Bula,Reich,F,1991-04-16,Australia,Single,Master’s Degree,bula.reich@lumen.com,2024-02-08,NaN,"""null""","""null""",4,34,3200


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
file_name = 'final project cleaning.xlsx'

In [ ]:
fact_df = pd.read_excel(file_name, sheet_name='FactEmployeeDetails (2)')
dim_df = pd.read_excel(file_name, sheet_name='DimEmployee1')

display(source_analysis)

In [ ]:
df = pd.merge(fact_df, dim_df[['EmployeeID', 'Gender', 'DateOfJoining']], on='EmployeeID', how='left')

In [ ]:
df['DateOfJoining'] = pd.to_datetime(df['DateOfJoining'], errors='coerce')
df['TerminationDate'] = pd.to_datetime(df['TerminationDate (New)'], errors='coerce')

In [ ]:
df['Is_Terminated'] = df['TerminationDate'].notnull().astype(int)

2.25% من إجمالي الموظفين استقالوا طواعية (VoluntaryExit == "Yes").

In [ ]:
vol_pct = (df[df['VoluntaryExit'] == 'Yes'].shape[0] / len(df)) * 100
print(f"Voluntary Attrition: {vol_pct:.2f}%")

Voluntary Attrition: 2.25%


عدد الاناث للذكور في العمل

In [ ]:
gender_distribution = df['Gender'].value_counts()
display(gender_distribution)

,count
Gender,
M,1228
F,1040


ما هو متوسط وقت التوظيف؟
المتوسط العام هو 38.46 يوم من لحظة الطلب حتى التعيين.

In [ ]:
print(f"Avg Time to Hire: {df['TimeToHire'].mean():.2f} days")

Avg Time to Hire: 38.46 days


In [ ]:
source_analysis = df.groupby('RecruitmentSource').agg(
    Count=('EmployeeID', 'count'),
    Left_Count=('Is_Terminated', 'sum'),
    Attrition_Rate=('Is_Terminated', lambda x: (x.mean() * 100).round(2))
)

In [ ]:
avg_cost_per_hire = df.groupby('RecruitmentSource')['CostPerHire'].mean().reset_index()
display(avg_cost_per_hire)

,RecruitmentSource,CostPerHire
0,1,3123.224568
1,2,3120.643939
2,3,3180.802792
3,4,3095.510836


In [ ]:
source_analysis['Attrition_Rate'] = source_analysis['Attrition_Rate'].astype(str) + '%'
display(source_analysis)

,Count,Left_Count,Attrition_Rate
RecruitmentSource,,,
1,521,21,4.03%
2,528,11,2.08%
3,573,21,3.66%
4,646,17,2.63%


In [ ]:
f = pd.merge(fact_df, dim_df[['EmployeeID', 'Gender', 'DateOfJoining']], on='EmployeeID', how='left')
df['TerminationDate'] = pd.to_datetime(df['TerminationDate (New)'], errors='coerce')
df['Is_Terminated'] = df['TerminationDate'].notnull().astype(int)

In [ ]:
source_map = {1: 'Job Board', 2: 'Referral', 3: 'Direct Application', 4: 'LinkedIn'}
df['SourceName'] = df['RecruitmentSource'].map(source_map)


In [ ]:
gender_attrition = df.groupby('Gender').agg(
    Total_Employees=('EmployeeID', 'count'),
    Left_Employees=('Is_Terminated', 'sum')
)

In [ ]:
gender_attrition['Attrition_Rate (%)'] = (gender_attrition['Left_Employees'] / gender_attrition['Total_Employees'] * 100).round(2)
print("\n1. معدل ترك العمل حسب الجنس:")
print(gender_attrition)


1. معدل ترك العمل حسب الجنس:
        Total_Employees  Left_Employees  Attrition_Rate (%)
Gender                                                     
F                  1040              36                3.46
M                  1228              34                2.77


In [ ]:
source_analysis = df.groupby('SourceName').agg(
    Hired_Count=('EmployeeID', 'count'),
    Left_Count=('Is_Terminated', 'sum'),
    Avg_TimeToHire=('TimeToHire', 'mean'),
    Avg_CostPerHire=('CostPerHire', 'mean')
)

In [ ]:
source_analysis['Attrition_Rate (%)'] = (source_analysis['Left_Count'] / source_analysis['Hired_Count'] * 100).round(2)

print("\n2. تحليل مصادر التوظيف:")
print(source_analysis[['Hired_Count', 'Left_Count', 'Attrition_Rate (%)']])


2. تحليل مصادر التوظيف:
                    Hired_Count  Left_Count  Attrition_Rate (%)
SourceName                                                     
Direct Application          573          21                3.66
Job Board                   521          21                4.03
LinkedIn                    646          17                2.63
Referral                    528          11                2.08


In [ ]:
avg_time = df['TimeToHire'].mean()
avg_cost = df['CostPerHire'].mean()

print(f"\n3. متوسط وقت التوظيف العام: {avg_time:.2f} يوم")
print(f"4. متوسط تكلفة التوظيف العامة: {avg_cost:,.2f} دولار")


3. متوسط وقت التوظيف العام: 38.46 يوم
4. متوسط تكلفة التوظيف العامة: 3,129.28 دولار


In [ ]:
voluntary_count = df[df['VoluntaryExit'] == 'Yes'].shape[0]
voluntary_pct = (voluntary_count / len(df)) * 100
print(f"\n5. الاستقالة الطوعية: {voluntary_count} موظف بنسبة ({voluntary_pct:.2f}%)")


5. الاستقالة الطوعية: 51 موظف بنسبة (2.25%)


In [ ]:
df = pd.merge(fact_df, dim_df[['EmployeeID', 'DateOfJoining']], on='EmployeeID', how='left')
df['DateOfJoining'] = pd.to_datetime(df['DateOfJoining'])
df['TerminationDate'] = pd.to_datetime(df['TerminationDate (New)'], errors='coerce')

In [ ]:
reference_date = df['TerminationDate'].max()
df['Tenure_Months'] = df.apply(
    lambda x: (x['TerminationDate'] - x['DateOfJoining']).days / 30.44
    if pd.notnull(x['TerminationDate'])
    else (reference_date - x['DateOfJoining']).days / 30.44, axis=1
)

In [ ]:
source_map = {1: 'Job Board', 2: 'Referral', 3: 'Direct Application', 4: 'LinkedIn'}
df['SourceName'] = df['RecruitmentSource'].map(source_map)

In [ ]:
retention_analysis = df.groupby('SourceName').agg(
    Total_Hires=('EmployeeID', 'count'),
    Avg_Tenure_Months=('Tenure_Months', 'mean')
).sort_values(by='Avg_Tenure_Months', ascending=False)

In [ ]:
print("--- تحليل معدل الاحتفاظ حسب مصدر التوظيف ---")
print(retention_analysis.round(2))

--- تحليل معدل الاحتفاظ حسب مصدر التوظيف ---
                    Total_Hires  Avg_Tenure_Months
SourceName                                        
Job Board                   521             178.77
Direct Application          573             177.68
Referral                    528             146.46
LinkedIn                    646             142.24


In [ ]:
avg_cost_per_hire = df['CostPerHire'].mean()

print(f"متوسط تكلفة التوظيف لكل موظف: {avg_cost_per_hire:,.2f} دولار")

متوسط تكلفة التوظيف لكل موظف: 3,129.28 دولار


In [ ]:
correlation_cost_tenure = df[['CostPerHire', 'Tenure_Months']].corr().iloc[0,1]

print(f"معامل الارتباط بين التكلفة ومدة البقاء: {correlation_cost_tenure:.4f}")

معامل الارتباط بين التكلفة ومدة البقاء: -0.0176


In [ ]:
if correlation_cost_tenure > 0.1:
    print("التحليل: نعم، هناك ميل لبقاء الموظفين ذوي التكلفة الأعلى لفترة أطول.")
elif correlation_cost_tenure < -0.1:
    print("التحليل: هناك علاقة عكسية، الموظفون الأقل تكلفة يبقون لفترة أطول.")
else:
    print("التحليل: لا توجد علاقة ملحوظة؛ التكلفة العالية لا تضمن بقاء الموظف لفترة أطول.")

التحليل: لا توجد علاقة ملحوظة؛ التكلفة العالية لا تضمن بقاء الموظف لفترة أطول.


In [ ]:
source_cost_analysis = df.groupby('SourceName').agg(
    Avg_Cost=('CostPerHire', 'mean'),
    Avg_Tenure_Months=('Tenure_Months', 'mean')
).round(2)

print(source_cost_analysis)

                    Avg_Cost  Avg_Tenure_Months
SourceName                                     
Direct Application   3180.80             177.68
Job Board            3123.22             178.77
LinkedIn             3095.51             142.24
Referral             3120.64             146.46
